In [1]:
import sys
# sys.path.insert(0, '../../Utilities/')

# from jax.example_libraries import optimizers
import jax.numpy as jnp
import numpy as np
from jax import grad, jit, vmap, value_and_grad, hessian
from jax import random
from functools import partial
import flax.linen as nn
import jax
import optax
import pickle
from datetime import datetime
import os
import numpy as np
from scipy import sparse
import time
from jax import grad, hessian, jit, vmap, pmap
from jax.tree_util import tree_leaves
from datetime import datetime
import matplotlib.pyplot as plt
import pickle
import flax.linen as nn
import optax
import optax.tree_utils as otu

from minmax_opt_class_mgnn import MinMaxOptProb

In [2]:
eps0 = 1e-4
def smooth_sign(x, eps = eps0):
    return 2*jax.nn.sigmoid(x/eps)-1

# def u0(x):
#     return 0.5 * (smooth_sign(x - 0.4) - smooth_sign(x - 0.6))

# def u1(x):
#     return 0

# def u2(x):
#     return 0


# Define boundary and initial conditions
def u0(x):
    # return 2*smooth_sign(x) - 1
    return -0.5*(jnp.sign(x) - 1)
    # return -jnp.sin(jnp.pi*x)
def u_exact(tx):
    t, x = tx[0], tx[1]
    return u0(x-0.5*t)
vec_u_exact = vmap(u_exact)

# Boundary at x=-1
def u1(x):
    return 1

# Boundary at x=1
def u2(x):
    return 0



def res_visc(u_quad, u_t, u_x, u_xx, nu, u_exact=0):
    # print(f"u_quad shape: {u_quad.shape}")
    # print(f"u2_x shape: {u2_x.shape}")
    # print(f"u_xx shape: {u_xx.shape}")
    return u_t + u_quad*u_x - nu*u_xx
    # return u_t + 0.8*u_x - nu*u_xx
    # return jnp.zeros_like(u_quad)
    # return u_quad - u_exact

def res(u_quad, u_t, u_x):
    return u_t + u_quad * u_x
    # return u_t + 0.8*u_x

# Initialize separate loss history lists for each stage
pre_loss_int_history = []
pre_loss_bc_history = []
train_loss_int_history = []
train_loss_bc_history = []

# Initialize the optimization problem
optProb = MinMaxOptProb(
    N_t=16,
    N_x=32,
    lr_theta=1e-3,
    N_bc=100,
    N_ic=200,
    t_min=0, t_max=1, x_min=-1, x_max=1,
    hidden_dims_list=[(40, 40), (40, 40), (40, 40)],
    seed=0,
    u0=u0, u1=u1, u2=u2,
    nu=0.01,
    res_visc=res_visc,
    res=res,    
)

In [3]:
optProb.train_process()


Starting Train Stage with MGDL


wandb: Currently logged in as: haoli1 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Stage 1: Sequential Grade Training

Training Grade 0
Grade 0, Iter 0, lambda_ic: 9.79249859, lambda_bc: 9.48744297, lambda_r: 1.04955173
Grade 0, Iter 0, loss_eqn: 0.94238180, loss_ibc: 1.23234665

Training Grade 1
Grade 1, Iter 0, lambda_ic: 9.68491364, lambda_bc: 8.77703094, lambda_r: 1.15932906
Grade 1, Iter 0, loss_eqn: 0.83347988, loss_ibc: 1.33065140

Training Grade 2
Grade 2, Iter 0, lambda_ic: 11.17939949, lambda_bc: 8.51609612, lambda_r: 1.16882515
Grade 2, Iter 0, loss_eqn: 1.86526060, loss_ibc: 0.63736641

Stage 2: Refining Last 2 Grades
Stage 2, Iter 0, lambda_ic: 11.07511234, lambda_bc: 8.18957806, lambda_r: 1.19260859
Stage 2, Iter 0, loss_eqn: 1.30948377, loss_ibc: 0.53230435
Stage 2, Iter 100, loss_eqn: 0.29991573, loss_ibc: 0.11135405
Stage 2, Iter 200, loss_eqn: 0.33454108, loss_ibc: 0.04084098
Stage 2, Iter 300, loss_eqn: 0.42017779, loss_ibc: 0.02034761
Stage 2, Iter 400, loss_eqn: 0.42345691, loss_ibc: 0.01604837
Stage 2, Iter 500, loss_eqn: 0.51708192, loss_ibc: 

grade_0_loss_eqn,▁
grade_0_loss_ibc,▁
grade_1_loss_eqn,▁
grade_1_loss_ibc,▁
grade_2_loss_eqn,▁
grade_2_loss_ibc,▁
second_stage_loss,▆█▇▅▇▇▅▄▅▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
second_stage_loss_ibc,█▄▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇█████
grade_0_loss_eqn,0.94238
grade_0_loss_ibc,1.23235


FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\train_model_20250706_160325.pkl'

In [ ]:
key = jax.random.key(0)
X = 100*jax.random.normal(key, (10000,2))
net_u = lambda tx: optProb.model.forward(tx).squeeze()
u_eval = jit(vmap(net_u, in_axes=0))
u_eval1 = jit(net_u)

In [5]:
params = optProb.model.mlps[0].params
optProb.u_eval(params, optProb.tx_t0)

Array([0.2923711 , 0.29203176, 0.29168776, 0.2913392 , 0.29098585,
       0.29062802, 0.2902654 , 0.2898982 , 0.28952628, 0.28914982,
       0.2887686 , 0.28838274, 0.28799227, 0.28759715, 0.28719744,
       0.28679317, 0.2863842 , 0.2859706 , 0.2855525 , 0.28512976,
       0.28470236, 0.28427052, 0.28383407, 0.28339309, 0.28294763,
       0.28249764, 0.28204322, 0.2815842 , 0.28112078, 0.28065288,
       0.28018066, 0.27970397, 0.27922288, 0.27873752, 0.27824774,
       0.27775362, 0.2772553 , 0.27675265, 0.2762458 , 0.2757347 ,
       0.2752194 , 0.2747    , 0.2741764 , 0.27364865, 0.27311692,
       0.27258104, 0.27204132, 0.27149743, 0.27094972, 0.270398  ,
       0.26984248, 0.26928294, 0.26871967, 0.2681527 , 0.2675819 ,
       0.26700744, 0.26642928, 0.26584744, 0.26526204, 0.2646731 ,
       0.26408058, 0.2634847 , 0.26288524, 0.2622824 , 0.26167625,
       0.26106676, 0.26045406, 0.25983813, 0.25921902, 0.25859672,
       0.25797138, 0.2573429 , 0.25671145, 0.256077  , 0.25543

In [6]:
optProb.loss_fn_ic(params)

Array(0.28839403, dtype=float32)

In [9]:
list(params)[0]

Array(0.1, dtype=float32, weak_type=True)

In [9]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = os.path.join('..', 'data', f'train_model_{timestamp}.pkl')

In [6]:
pwd

'C:\\Users\\hao6li\\Projects\\NN-HCL\\Code'

In [7]:
ls

 Volume in drive C has no label.
 Volume Serial Number is C860-DE6E

 Directory of C:\Users\hao6li\Projects\NN-HCL\Code

07/07/2025  08:58 PM    <DIR>          .
07/03/2025  05:58 PM    <DIR>          ..
07/03/2025  05:58 PM             6,148 .DS_Store
07/05/2025  10:15 PM    <DIR>          .ipynb_checkpoints
07/06/2025  03:37 PM    <DIR>          __pycache__
07/03/2025  05:58 PM           102,064 0-Burgers.ipynb
07/03/2025  05:58 PM         1,001,248 Burgers_1D_rarefaction_shock.ipynb
07/03/2025  06:28 PM           780,087 Burgers_1D_rarefaction_wave.ipynb
07/03/2025  05:58 PM           298,110 Burgers_1D_standing_shock.ipynb
07/05/2025  10:19 PM           850,088 Burgers_1D_traveling_shock.ipynb
07/03/2025  05:58 PM         2,977,085 Burgers_bdry_int_loss.ipynb
07/03/2025  05:58 PM           829,005 Burgers_rarefaction_shock_test.ipynb
07/03/2025  05:58 PM    <DIR>          data
07/03/2025  05:58 PM           744,515 HCL_1D.ipynb
07/03/2025  05:58 PM         1,612,266 HCL_1D_triangle

In [18]:
optProb.save_model(model_path)

In [16]:
os.makedirs(os.path.dirname(model_path), exist_ok=True)

In [15]:
model_path = os.path.join('data', f'train_model_{timestamp}.pkl')

In [13]:
cd ..

D:\minconda\envs\jax-env\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


C:\Users\hao6li\Projects\NN-HCL


In [19]:
model_path


'data\\train_model_20250707_211627.pkl'